# 05 · Galeria de faces na web e desafio

**Pré-requisito:** notebooks 00 a 04 (as imagens da FEI já estão no `faces_db`).

**Nesta etapa:** o **Exercício 8** (app Streamlit com a galeria do banco de imagens) e um **desafio** que liga banco de dados e machine learning.

> **Sobre os dados:** As imagens da FEI são liberadas só para pesquisa, sem redistribuição. Ao publicar o app, **restrinja o acesso** a convidados: no Streamlit Community Cloud, abra o app → **Share** → defina que só pessoas convidadas podem ver e convide o e-mail do professor.

In [ ]:
%pip install -q "pymongo[srv]" python-dotenv pillow numpy matplotlib requests streamlit

## Conexão com o seu banco no Atlas

A célula abaixo lê a sua URI do arquivo **`.env`** da pasta (criado no notebook 00, Etapa 6) e testa a conexão.

In [ ]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()                                   # lê o arquivo .env da pasta
client = MongoClient(os.environ["MONGO_URI"])
client.admin.command("ping")
print("Conectado ao MongoDB Atlas")

In [ ]:
import io, gridfs
from PIL import Image
db = client["faces_db"]
fs = gridfs.GridFS(db)
print(db.fs.files.count_documents({}), "imagens no banco")

---
## Agora vamos fazer o Exercício 8 · Galeria de faces na web

**Você vai usar:** `st.selectbox`, `st.number_input`, `count_documents`, `skip`/`limit`, `st.columns`, `st.image`, `st.file_uploader`, `fs.put`.

1. **Filtros:** expressão (todas, neutra, sorrindo) e número da pessoa (0 = todas).
2. **Galeria:** grade com **4 colunas**, **20 imagens por página**, com seletor de página.
3. **Upload:** `st.file_uploader` para enviar uma foto nova direto para o GridFS, com metadados preenchidos pelo usuário (pessoa e expressão). Não aceite nomes repetidos.
4. **Teste no seu computador** e depois **publique** (mesmo passo a passo do notebook 03, Parte 3, com um repositório novo, por exemplo `galeria-faces`). **Restrinja o acesso** e envie o link.

O app fica na pasta `galeria`. A primeira célula cria a pasta e o arquivo de segredos; a segunda grava o esqueleto, com os `TODO` para você completar.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.makedirs("galeria/.streamlit", exist_ok=True)
with open("galeria/.streamlit/secrets.toml", "w", encoding="utf-8") as f:
    f.write(f'MONGO_URI = "{os.environ["MONGO_URI"]}"\n')
print("Criados: galeria/  e  galeria/.streamlit/secrets.toml")

In [ ]:
%%writefile galeria/app.py
import io
import gridfs
import streamlit as st
from PIL import Image
from pymongo import MongoClient

st.set_page_config(page_title="Galeria de faces", layout="wide")

@st.cache_resource
def conectar():
    db = MongoClient(st.secrets["MONGO_URI"])["faces_db"]
    return db, gridfs.GridFS(db)

db, fs = conectar()
st.title("Galeria de faces · FEI")

# TODO 1: filtros -> monte o dicionário `filtro` usando "metadata.expressao" e "metadata.pessoa"
filtro = {}

# TODO 2: paginação e grade
#   total = db.fs.files.count_documents(filtro)
#   arquivos = db.fs.files.find(filtro).sort("metadata.pessoa", 1).skip(...).limit(20)
#   colunas = st.columns(4); colunas[i % 4].image(Image.open(io.BytesIO(fs.get(info["_id"]).read())), caption=...)

# TODO 3: upload com st.file_uploader e fs.put(dados, filename=..., metadata={...})

In [ ]:
%%writefile galeria/requirements.txt
streamlit
pymongo[srv]
pillow
numpy

**Para testar:** Terminal → New Terminal → `cd galeria` → `python -m streamlit run app.py`.

---
## Desafio extra · Qual é a face mais parecida?

Ligando banco de dados e machine learning: dada uma foto nova, encontrar no banco a face mais parecida.

**Ideia:**
1. **Vetorizar:** cada imagem vira um vetor de números (tons de cinza, 64 × 64 = 4096 valores entre 0 e 1). A função já está pronta abaixo.
2. **Guardar os vetores** numa coleção `vetores` (`file_id`, `filename`, `vetor` como lista), para não recalcular a cada busca.
3. **Comparar:** calcular a **distância euclidiana** entre o vetor da foto nova e todos os vetores do banco. A menor distância é a face mais parecida.
4. **Levar para o app** do Exercício 8, numa aba ou seção “Encontrar a face mais parecida”.

Para ir além: compare com a **distância de cosseno** ou reduza os vetores com **PCA** antes de comparar (as famosas *eigenfaces*).

In [ ]:
import numpy as np

def vetorizar(dados_bytes, tamanho=(64, 64)):
    img = Image.open(io.BytesIO(dados_bytes)).convert("L").resize(tamanho)
    return (np.asarray(img, dtype=np.float32) / 255.0).ravel()

# teste com a primeira imagem do banco
primeira = db.fs.files.find_one()
if primeira:
    v = vetorizar(fs.get(primeira["_id"]).read())
    print(primeira["filename"], "->", v.shape, v[:5])

In [ ]:
# Desafio 2: gravar os vetores na coleção db['vetores'] (sem duplicar)
# seu código aqui

In [ ]:
# Desafio 3: função mais_parecida(dados_bytes) -> (documento, distancia)
# dica: np.linalg.norm(matriz - alvo, axis=1)
# seu código aqui

In [ ]:
# Teste: use uma foto "b" (sorrindo) como consulta e veja se a mais parecida
# (ignorando a própria foto) é a "a" da mesma pessoa
# seu código aqui

---
## Checklist de entrega
- [ ] Coleção `escola.alunos` no Atlas (Exercícios 1 a 3)
- [ ] App “Secretaria online” publicado (Exercício 4)
- [ ] Faces da FEI no GridFS, sem duplicatas e com metadados (Exercícios 5 e 6)
- [ ] Consultas e galeria no notebook (Exercício 7)
- [ ] App “Galeria de faces” publicado com acesso restrito e link enviado (Exercício 8)
- [ ] (Extra) Busca pela face mais parecida